# 10b: Validity, more repetitions for the inconclusive configurations (gate G4)

Design decisions (recorded before the run, docs/PLAN.md)
- Plan Section 6.8 and gate G4: an interval that crosses delta is inconclusive at that number of repetitions, and inconclusive configurations get more repetitions. Notebook 10 left 35 of the 936 SCAX configurations inconclusive at 500 repetitions (all with the mixture prior; every point estimate at or below delta_1); none showed excess violations.
- Every configuration that notebook 10 left inconclusive (box certificates or policy) gets 4,500 more repetitions, 5,000 in all, ten times the first run. No other configuration is rerun and the number is not changed after seeing results.
- Same population, label setting, model, candidate sets, priors, targets and grid as notebook 10: each setting is rebuilt from the candidate cache of notebook 10 with the same seed, and its classifier hash and box count must equal those of results/10_settings.csv. New repetition seeds (5000 + 10 x setting index + calibration-size index), so the added repetitions are independent of the first 500.
- Verdicts on the pooled repetitions by the same three-way rule (one-sided 95% Clopper-Pearson limits), the excess test corrected for the same 936 configurations. Configurations still inconclusive are reported with their point estimates; frequencies near delta_1 are expected where one box carries most of the prior mass, because the union bound is then nearly tight.
- Repetitions are stored on Drive after each setting, so an interrupted run resumes at the next setting.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_array, write_json
from scax.data import planted
from scax.experiments.validity import build_setting, run_repetitions
from scax.metrics.validity import validity_verdict

SEED = 42
seed_everything(SEED)
R_EXTRA = 4500                 # more repetitions per inconclusive configuration (5,000 in all)
XP = pd.read_parquet(paths.DERIVED / 'planted_population.parquet')
REC04 = json.load(open(f'{paths.RESULTS}/04_population_summary.json'))
assert sha256_array(XP.to_numpy()) == REC04['population_matrix_sha256'], 'population differs from notebook 04'
P4_B0 = planted.p4_intercept(XP)
S10 = json.load(open(f'{paths.RESULTS}/10_validity_summary.json'))
MODEL, N_TRAIN, N2, DELTA2, R_FIRST = S10['model'], S10['n_train'], S10['n2'], S10['delta2'], S10['repetitions']
M_MAX, K = max(S10['grid']['M']), S10['configurations_scax']
LAM = float([p for p in S10['grid']['priors'] if p.startswith('mix')][0][3:])
SETTINGS = [(r, e) for r in ('P1', 'P2', 'P3') for e in planted.NOISE_LEVELS] + [('P4', 0.0)]   # the order of notebook 10 (seeds)
KEYS = ['rule', 'eps', 'n1', 'delta1', 'M', 'prior']
V = pd.read_csv(f'{paths.RESULTS}/10_validity.csv')
SET10 = pd.read_csv(f'{paths.RESULTS}/10_settings.csv')
INC = V[(V.prior != 'naive') & ((V.V1_verdict == 'inconclusive') | (V.V2_verdict == 'inconclusive'))].copy()
for d in (V, SET10, INC):
    d['eps'] = d.eps.round(4)
OUT_DIR = paths.DERIVED / 'validity'


def targets(rule, eps):
    if rule == 'P4':
        return {0: 0.05, 1: 0.70}, 0.05
    a = round(eps + 0.05, 10)
    return {0: a, 1: a}, a


print(f"{len(INC)} inconclusive configurations of {K} (priors {INC.prior.unique().tolist()}) in "
      f"{INC.groupby(['rule', 'eps']).ngroups} label settings; {R_EXTRA} more repetitions each")
print(INC.groupby(['rule', 'eps']).size().to_string())

## More repetitions, setting by setting

In [ ]:
EXTRA, SECONDS = [], {}
for s_i, (rule, eps) in enumerate(SETTINGS):
    sub = INC[(INC.rule == rule) & np.isclose(INC.eps, eps)]
    if sub.empty:
        continue
    tag = f'{rule}_eps{eps:.2f}'
    f = OUT_DIR / f'{tag}_reps_more_{R_EXTRA}.parquet'
    if f.exists():
        EXTRA.append(pd.read_parquet(f)); print(tag, 'loaded from Drive'); continue
    t0 = time.time()
    alpha, alpha_sel = targets(rule, eps)
    st, diag = build_setting(XP, rule, eps, P4_B0, MODEL, N_TRAIN, M_MAX, SEED + s_i, alpha, alpha_sel, lam=LAM,
                             cache_path=paths.DERIVED / 'cache' / f'10_{tag}.pkl', z3_n=0, log=lambda *a: None)
    ref = SET10[(SET10.rule == rule) & np.isclose(SET10.eps, eps)].iloc[0]
    assert diag['classifier_sha256'] == ref.classifier_sha256 and int(diag['boxes']) == int(ref.boxes), \
        f'{tag}: rebuilt setting differs from notebook 10'
    t1 = time.time()
    parts = []
    for j, (n1, g) in enumerate(sub.groupby('n1')):
        r = run_repetitions(st, [int(n1)], sorted(g.M.unique()), sorted(g.delta1.unique()), sorted(g.prior.unique()),
                            R_EXTRA, N2, DELTA2, seed=5000 + 10 * s_i + j)
        parts.append(r.merge(g[['n1', 'delta1', 'M', 'prior']], on=['n1', 'delta1', 'M', 'prior']))   # only these configurations
    reps = pd.concat(parts, ignore_index=True)
    reps.insert(0, 'eps', round(eps, 4)); reps.insert(0, 'rule', rule)
    assert reps.groupby(['n1', 'delta1', 'M', 'prior']).size().eq(R_EXTRA).all() and len(reps) == R_EXTRA * len(sub)
    reps.to_parquet(f, index=False)
    EXTRA.append(reps)
    SECONDS[tag] = {'rebuild': t1 - t0, 'repetitions': time.time() - t1}
    print(f'{tag}: {len(sub)} configurations, rebuilt in {t1 - t0:.0f}s, {R_EXTRA} repetitions in {time.time() - t1:.0f}s', flush=True)

## Pooled verdicts

In [ ]:
X = pd.concat(EXTRA, ignore_index=True)
AGG = X.groupby(KEYS).agg(reps_extra=('V1', 'size'), V1_extra=('V1', 'sum'), V2_extra=('V2', 'sum'),
                          cert_rate_extra=('cert_rate', 'mean'), policy_certified_extra=('policy_certified', 'mean')).reset_index()
OUT = INC[KEYS + ['reps', 'V1', 'V2', 'V1_rate', 'V1_upper', 'V1_verdict', 'V2_verdict', 'cert_rate']].rename(columns={
    'reps': 'reps_first', 'V1': 'V1_first', 'V2': 'V2_first', 'V1_rate': 'V1_rate_first', 'V1_upper': 'V1_upper_first',
    'V1_verdict': 'V1_verdict_first', 'V2_verdict': 'V2_verdict_first', 'cert_rate': 'cert_rate_first'}).merge(AGG, on=KEYS, how='left')
assert OUT.reps_extra.notna().all(), 'a configuration has no added repetitions'
OUT['V1_rate_extra'] = OUT.V1_extra / OUT.reps_extra
OUT['reps_total'] = (OUT.reps_first + OUT.reps_extra).astype(int)
OUT['V1_total'] = (OUT.V1_first + OUT.V1_extra).astype(int)
OUT['V2_total'] = (OUT.V2_first + OUT.V2_extra).astype(int)
v1 = [validity_verdict(int(a), int(n), float(d), k_configs=K) for a, n, d in OUT[['V1_total', 'reps_total', 'delta1']].values]
v2 = [validity_verdict(int(a), int(n), DELTA2, k_configs=K) for a, n in OUT[['V2_total', 'reps_total']].values]
OUT['V1_rate_total'] = [v['rate'] for v in v1]; OUT['V1_upper_total'] = [v['upper'] for v in v1]
OUT['V1_lower_total'] = [v['lower'] for v in v1]; OUT['V1_verdict_total'] = [v['verdict'] for v in v1]
OUT['V2_rate_total'] = [v['rate'] for v in v2]; OUT['V2_upper_total'] = [v['upper'] for v in v2]
OUT['V2_verdict_total'] = [v['verdict'] for v in v2]
pd.set_option('display.width', 250)
print(OUT[['rule', 'eps', 'n1', 'delta1', 'M', 'prior', 'V1_first', 'V1_rate_first', 'V1_extra', 'V1_rate_extra', 'V1_rate_total',
           'V1_upper_total', 'V1_verdict_total', 'V2_verdict_total']].round(4).to_string(index=False))
first, total = OUT.V1_verdict_first.value_counts().to_dict(), OUT.V1_verdict_total.value_counts().to_dict()
print(f'\nbox certificates, these {len(OUT)} configurations: after {R_FIRST} repetitions {first}; after {int(OUT.reps_total.max())} {total}')
print('policy:', OUT.V2_verdict_total.value_counts().to_dict())
left = OUT[OUT.V1_verdict_total == 'inconclusive']
if len(left):
    print(f'{len(left)} still inconclusive; point estimates at or below delta_1: {bool((left.V1_rate_total <= left.delta1).all())}')

In [ ]:
OUT.to_csv(f'{paths.RESULTS}/10b_validity_more_repetitions.csv', index=False)
write_json(f'{paths.RESULTS}/10b_validity_more_repetitions.json', {
    'seed': SEED, 'repetition_seeds': '5000 + 10 x setting index + calibration-size index', 'repetitions_first': R_FIRST,
    'repetitions_added': R_EXTRA, 'configurations': len(OUT), 'configurations_for_multiplicity': K,
    'V1_verdicts_first': first, 'V1_verdicts_total': total, 'V2_verdicts_total': OUT.V2_verdict_total.value_counts().to_dict(),
    'excess': int((OUT.V1_verdict_total == 'excess').sum() + (OUT.V2_verdict_total == 'excess').sum()),
    'still_inconclusive': int((OUT.V1_verdict_total == 'inconclusive').sum()), 'seconds': SECONDS})
print('written: results/10b_validity_more_repetitions.csv, 10b_validity_more_repetitions.json')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "10b: validity, 4,500 more repetitions (5,000 in all) for the configurations notebook 10 left inconclusive, pooled verdicts with the same multiplicity correction (gate G4)"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)